# Connection Capacity and Ephemeral Port Exhaustion

Companion notebook for the [Connection Limits and Capacity](https://codemia.ai/courses/networking_for_backend_engineers/net_connection_limits_and_capacity) lesson on Codemia.

**What you will build**: the arithmetic that decides how many connections a host can open per
second, why that limit arrives long before the CPU or the bandwidth does, and what connection
reuse buys. Everything here is a small number of divisions, which is exactly why it is worth
doing once by hand: the limit is knowable in advance and almost always discovered in production
instead.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DEFAULT_PORT_RANGE = (32768, 60999)        # the usual Linux default
DEFAULT_PORTS = DEFAULT_PORT_RANGE[1] - DEFAULT_PORT_RANGE[0] + 1
TIME_WAIT_SECONDS = 60                     # fixed at 2 x MSL on most stacks

print(f'ephemeral ports available by default: {DEFAULT_PORTS:,}')

## Concept recap

A connection is identified by four values: source address, source port, destination address,
destination port. The client picks the source port from a range, and when the connection closes the
client side holds that port in a waiting state for a fixed interval, so that a late packet from the
old connection cannot be mistaken for part of a new one.

The consequence is an arithmetic limit. If the destination is a single address and port, the number
of connections a source address can establish per second is the size of the port range divided by
the waiting interval. Not per minute, not under load: always, and no amount of hardware changes it.

## The core limit

In [ ]:
def connections_per_second(ports=DEFAULT_PORTS, time_wait=TIME_WAIT_SECONDS, destinations=1):
    """Each closed connection parks a port for `time_wait`; distinct destinations get their own range."""
    return ports * destinations / time_wait

print(f'one destination, default range  -> {connections_per_second():>9,.0f} new connections/s')
print(f'widened range (1024-65535)      -> {connections_per_second(ports=64512):>9,.0f} /s')
print(f'8 backend addresses             -> {connections_per_second(destinations=8):>9,.0f} /s')
print(f'8 backends, widened range       -> {connections_per_second(ports=64512, destinations=8):>9,.0f} /s')

Around 470 new connections per second to a single destination, with the defaults. A service doing
2,000 requests per second with a fresh connection per request cannot work, and it will not fail
with a capacity error: it will fail with connection timeouts and address-in-use errors that look
like a problem on the far side.

In [ ]:
twaits = np.array([1, 5, 15, 30, 60, 120, 240])

plt.figure(figsize=(9, 5))
for ports, label in [(DEFAULT_PORTS, 'default range (28k ports)'),
                     (64512, 'widened range (64k ports)')]:
    plt.plot(twaits, [connections_per_second(ports, t) for t in twaits], marker='o', label=label)

plt.axvline(60, color='grey', linestyle=':', linewidth=1)
plt.text(62, 1000, 'the usual 60s', color='grey', fontsize=9)
plt.xlabel('time a closed port stays reserved (seconds)')
plt.ylabel('sustainable new connections/s to one destination')
plt.yscale('log')
plt.title('Two levers, one of which you should not pull')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

Both levers are visible and only one is safe. Widening the port range is free. Shortening the
waiting interval reintroduces exactly the hazard the interval exists to prevent, and on a lossy
path with retransmissions in flight that hazard is not theoretical.

The lever that is not on this plot is the one to reach for: stop closing connections.

## What reuse buys

In [ ]:
def ports_needed(requests_per_second, requests_per_connection, time_wait=TIME_WAIT_SECONDS):
    """Connections opened per second x the interval a closed port is held."""
    connections_per_second = requests_per_second / requests_per_connection
    return connections_per_second * time_wait

rps = 2_000
print(f'{rps:,} requests/s to one destination, {DEFAULT_PORTS:,} ports available')
print(f"{'requests per connection':>26}{'ports in use':>15}{'headroom':>12}")
for reuse in [1, 2, 10, 100, 1_000, 10_000]:
    need = ports_needed(rps, reuse)
    state = 'exhausted' if need > DEFAULT_PORTS else f'{DEFAULT_PORTS/need:>6.1f}x'
    print(f'{reuse:>26,}{need:>15,.0f}{state:>12}')

A single keep-alive connection carrying a hundred requests moves the problem out of sight. This is
the whole argument for connection pooling, and it is worth noting what the numbers say about pool
*size*: the pool does not need to be large, it needs to be long-lived. Ten connections that last an
hour serve the same traffic as thousands that last a second, and they occupy ten ports instead of
exhausting the range.

## When a proxy sits in the middle

In [ ]:
def nat_capacity(source_addresses, destinations, ports=DEFAULT_PORTS, time_wait=TIME_WAIT_SECONDS):
    """A translating device rewrites many clients onto its own few addresses."""
    return source_addresses * destinations * ports / time_wait

print(f"{'translator addresses':>22}{'destinations':>14}{'new conns/s':>14}")
for addresses in [1, 2, 8, 32]:
    for destinations in [1, 4]:
        print(f'{addresses:>22}{destinations:>14}{nat_capacity(addresses, destinations):>14,.0f}')

This is the case that surprises people. A cluster of two hundred pods behind one translating address
shares a single port range, so the limit applies to the cluster as a whole rather than to each pod.
The per-pod traffic looks modest, the aggregate hits the wall, and the failure appears on whichever
pod happens to ask next. Adding pods makes it worse.

Note the destinations column: spreading the same traffic over four backend addresses multiplies the
capacity, because the tuple includes the destination. Resolving a service name to several addresses
rather than one is a capacity decision, not only an availability one.

## Variations: find your own ceiling

In [ ]:
def diagnose(requests_per_second, requests_per_connection, destinations, source_addresses=1,
             ports=DEFAULT_PORTS, time_wait=TIME_WAIT_SECONDS):
    capacity = nat_capacity(source_addresses, destinations, ports, time_wait)
    opened = requests_per_second / requests_per_connection
    return opened, capacity, opened / capacity

cases = [
    ('no pooling, single backend',          2_000,     1, 1, 1),
    ('no pooling, 8 backends',              2_000,     1, 8, 1),
    ('pooled, single backend',              2_000,   500, 1, 1),
    ('200 pods behind one NAT address',    20_000,     1, 4, 1),
    ('200 pods, pooled, 4 backends',       20_000,   500, 4, 1),
]

print(f"{'case':<34}{'opened/s':>11}{'capacity/s':>13}{'utilisation':>13}")
for name, rps, reuse, dests, addrs in cases:
    opened, capacity, util = diagnose(rps, reuse, dests, addrs)
    flag = '  OVER' if util > 1 else ''
    print(f'{name:<34}{opened:>11,.0f}{capacity:>13,.0f}{util:>12.1%}{flag}')

Two of these are over the line and both are fixed by the same change. The useful habit is to run
this calculation before a load test rather than after: if the utilisation column is above about 50%
you are one traffic spike away from connection errors, and the error you will see says nothing
about ports.

## What to try next

- Check the real numbers on a host: the port range is in `/proc/sys/net/ipv4/ip_local_port_range`,
  and `ss -s` will show you how many sockets are in the waiting state right now. Put them into
  `connections_per_second` and compare against your request rate.
- Add the listening side. A server has its own limits, from the accept queue to the file-descriptor
  limit, and they are a different calculation with a different failure signature. The connection
  limits lesson covers both.
- Model a retry storm. A client that retries on timeout opens more connections exactly when the
  port range is tightest, which is how a small incident becomes a sustained one.